In [1]:
import MySQLdb

In [2]:
class Word:
    # Word 클래스는 단어 하나의 정보를 저장하는 클래스이다.
    # 영어 단어, 뜻, 레벨을 하나의 객체로 묶어서 관리한다.
    def __init__(self, eng, kor, lev=1):
        # 입력 받은 값들은 각각 self.~에 저장된다.
        # 이때 바로 변수가 저장되는 것이 아니라, 아래에 있는 setter가 자동으로 호출된다.
        self.eng = eng
        self.kor = kor
        self.lev = lev
    
    def __repr__(self):
        # 객체를 출력하거나 확인할 때 보기 좋은 문자열 형태로 표현해주는 메서드이다.
        # 예를 들어 Word("apple", "사과", 1)을 출력하면 객체 주소 대신 단어 정보가 보이게 한다.
        return f"Word(eng='{self.eng}', kor='{self.kor}', lev={self.lev})"
    
    @property # eng 값을 가져올 때 사용하는 getter를 만든다.
    def eng(self):
        # eng getter 메서드이다.
        # 외부에서 self.eng를 읽으려고 할 때 실행된다.
        return self.__eng
    
    @eng.setter
    # eng 값을 저장하거나 수정할 때 사용하는 setter를 만든다.
    # self.eng = 값 형태로 대입할 때 이 메서드가 실행된다.

    def eng(self, eng):
        # eng setter 메서드이다.
        # 영어 단어 값을 저장하기 전에 검사를 할 수 있다.
        if not eng: 
            raise ValueError("영어 단어는 비워둘 수 없습니다")
        self.__eng = eng
        # 검사를 통과한 영어 단어를 내부 변수 __eng에 저장한다.

    # 아래의 코드 또한 eng의 getter, setter의 구성과 동일하다.
    @property
    def kor(self):
        return self.__kor
    
    @kor.setter
    def kor(self, kor):
        if not kor:
            raise ValueError("뜻을 비워둘 수 없습니다")
        self.__kor = kor
    
    @property
    def lev(self):
        return self.__lev
    
    @lev.setter
    def lev(self, lev):
        lev = int(lev)
        if lev < 1:
            raise ValueError("레벨은 1 이상이어야 합니다")
        self.__lev = lev

In [3]:
class WordsDAO:
    # DAO : Data Access Object
    # 데이터베이스에 직접 접근하는 역할을 담당한다.
    def __init__(self):
        self.db = None
        # DB 연결 객체를 저장할 변수이다.
        # 처음에는 아직 DB에 연결하지 않았으므로 None으로 둔다.
    
    def connect(self): # MySQL 데이터베이스에 연결하는 메서드이다.
        self.db = MySQLdb.connect(host='localhost', user='apple', password='1111', db='ai', charset='utf8')
        # 연결된 DB 객체를 self.db에 저장한다.
    
    def disconnect(self): # 데이터베이스 연결을 종료하는 메서드이다.
        if self.db: # 만약 DB연결 객체가 있다면
            self.db.close() # close 실행
    
    def insert(self, word): # 단어 하나를 DB에 추가하는 메서드이다.
        self.connect() # 위의 connect() 함수 실행
        cur = self.db.cursor()
        # SQL을 실행할 cursor 객체를 만든다.
        # cursor는 SQL 명령을 DB에 전달하는 역할을 한다.
        sql = "insert into voca (eng, kor, lev) values (%s, %s, %s)"
        data = (word.eng, word.kor, word.lev)
        # Word 객체에서 영어 단어, 뜻, 레벨을 꺼낸다.
        cur.execute(sql, data) # SQL문 실행
        self.db.commit()
        cur.close()
        self.disconnect() # 위의 disconnect() 함수 실행

    def select_all(self): # DB에 저장된 모든 단어를 조회하는 메서드
        self.connect()
        cur = self.db.cursor(MySQLdb.cursors.DictCursor)
        # DictCursor를 사용하여 조회 결과를 딕셔너리 형태 나오게 한다.
        sql = "select eng, kor, lev from voca order by eng asc"
        # voca 테이블에서 eng, kor, lev 컬럼을 조회
        cur.execute(sql) # SQL문 실행
        rows = cur.fetchall() # 조회된 모든 결과를 가져오고 rows에 저장한다.
        cur.close()
        self.disconnect()
        return rows # 조회한 전체 단어 목록을 반환한다.
    
    def search_all(self, eng): # 입력한 글자가 포함된 단어들을 검색하는 메서드
        self.connect()
        cur = self.db.cursor(MySQLdb.cursors.DictCursor)
        # 조회 결과를 딕셔너리 형태로 받기 위해 DictCursor를 사용한다.
        sql = "select eng, kor, lev from voca where eng like concat('%%', %s, '%%')"
        # concat('%%', %s, '%%')는 앞뒤에 %를 붙여 부분 검색을 하겠다는 뜻이다.
        # 파이썬 문자열 안에서 % 자체를 표현하기 위해 %%를 사용했다.
        cur.execute(sql, (eng,)) # SQL문 실행
        # 튜플에서 값이 하나일 때는 쉼표를 꼭 붙여야 한다.
        rows = cur.fetchall() # 조회된 모든 결과를 가져오고 rows에 저장한다.
        cur.close()
        self.disconnect()
        return rows # 검색 결과 목록을 반환

    def search(self, eng): # 정확히 일치하는 영어 단어 하나를 검색하는 메서드
        self.connect()
        cur = self.db.cursor(MySQLdb.cursors.DictCursor)
        # 조회 결과를 딕셔너리 형태로 받기 위해 DictCursor를 사용
        sql = "select eng, kor, lev from voca where eng=%s"
        cur.execute(sql, (eng,)) # SQL문 실행
        row = cur.fetchone()
        # 조회 결과 중 첫 번째 행 하나만 가져온다.
        # 결과가 없으면 None이 반환된다.
        cur.close()
        self.disconnect()
        return row
        # 검색된 단어 하나를 반환한다.
        # 없으면 None을 반환한다.

    def update(self, word): # 기존 단어의 뜻과 레벨을 수정하는 메서드
        self.connect()
        cur = self.db.cursor() # SQL 실행을 위한 cursor를 만든다.
        sql = "update voca set kor=%s, lev=%s where eng=%s"
        # voca 테이블에서 eng가 일치하는 단어를 찾아 kor와 lev를 수정하는 SQL문이다.
        # 영어 단어는 기준값으로 사용하고, 뜻과 레벨만 수정한다.
        data = (word.kor, word.lev, word.eng)
        # SQL의 %s 자리에 들어갈 데이터를 튜플로 만든다.
        result = cur.execute(sql, data) # SQL문 실행
        # result에는 영향을 받은 행의 개수가 들어간다.
        # 수정 성공 시 보통 1이 들어가고, 수정된 행이 없으면 0이 들어간다.
        self.db.commit()
        cur.close()
        self.disconnect()
        return result
        # 수정된 행의 개수를 반환한다.

    def delete(self, eng): # 입력받은 영어 단어를 DB에서 삭제하는 메서드
        self.connect()
        cur = self.db.cursor()
        sql = "delete from voca where eng = %s"
        result = cur.execute(sql, (eng,)) # SQL문 실행
        # 삭제된 행의 개수가 result에 저장된다.
        self.db.commit()
        cur.close()
        self.disconnect()
        return result
        # 삭제된 행의 개수를 반환한다.

In [4]:
class WordsService:
    # WordsService 클래스는 실제 프로그램의 기능을 담당
    # 사용자 입력을 받고, DAO를 이용해서 DB 작업을 요청한다.
    def __init__(self):
        self.dao = WordsDAO()
        # DB 작업을 담당할 WordsDAO 객체를 생성

    def insert_word(self): # 사용자가 입력한 단어를 등록하는 메서드
        eng = input("단어를 입력하세요: ")
        kor = input("뜻을 입력하세요: ")
        lev = input("레벨을 입력하세요: ")

        word = Word(eng, kor, lev)
        # 입력받은 eng, kor, lev를 이용해 Word 객체를 만든다.
        # 이때 Word 클래스 내부에서 값 검사가 이루어진다.
        self.dao.insert(word)
        # DAO의 insert 메서드를 호출해서 DB에 단어를 저장한다.
        print("단어가 등록되었습니다")

    def print_all(self): # DB에 저장된 모든 단어를 출력하는 메서드
        words = self.dao.select_all()
        # DAO의 select_all 메서드를 호출해서 전체 단어 목록을 가져온다.
        if not words:
            print("등록된 단어가 없습니다")
            return
            # 더 이상 출력할 것이 없으므로 메서드를 종료
        for word in words:
            # words 리스트 안에 있는 단어 정보를 하나씩 꺼낸다.
            # word는 딕셔너리 형태이다.
            print(f"{word['eng']} 뜻:{word['kor']}, 레벨:{word['lev']}")
    
    def search_words(self): # 사용자가 입력한 글자가 포함된 단어를 검색하는 메서드
        eng = input("검색할 단어를 입력하세요: ")
        # 검색할 영어 단어 또는 일부 글자를 입력받는다.
        words = self.dao.search_all(eng)
        # DAO의 search_all 메서드를 호출해서 포함 검색을 수행한다.
        if not words:
            print("찾는 단어가 없습니다")
            return # 더 이상 출력할 것이 없으므로 메서드를 종료
        for word in words:
            print(f"{word['eng']} 뜻:{word['kor']}, 레벨:{word['lev']}")

    def edit_word(self): # 등록된 단어의 뜻과 레벨을 수정하는 메서드
        eng = input("수정할 단어를 입력하세요: ")
        # 수정할 영어 단어를 입력받는다.
        word = self.dao.search(eng)
        # 입력한 단어가 DB에 존재하는지 정확히 검색한다.
        if not word:
            print("수정할 단어가 없습니다")
            return # 수정할 대상이 없으므로 메서드를 종료
        kor = input("새로운 뜻을 입력하세요: ")
        lev = input("새로운 레벨을 입력하세요: ")
        word = Word(eng, kor, lev)
        # 기존 영어 단어 eng에 새로운 뜻과 레벨을 넣어 Word 객체를 만든다.
        # 이때 kor와 lev 값 검사가 이루어진다.
        result = self.dao.update(word)
        # DAO의 update 메서드를 호출해서 DB 내용을 수정한다.
        # result에는 수정된 행의 개수가 들어간다.
        if result > 0: # 수정된 행의 개수가 0보다 크면 수정 성공
            print('수정되었습니다')
        else:
            print('수정에 실패했습니다')
    
    def delete_word(self): # 등록된 단어를 삭제하는 메서드
        eng = input("삭제할 단어를 입력하세요: ")
        word = self.dao.search(eng)
        # 입력한 단어가 DB에 존재하는지 먼저 검색한다.
        if not word:
            print("삭제할 단어가 없습니다")
            return # 삭제할 대상이 없으므로 메서드를 종료
        result = self.dao.delete(eng)
        # DAO의 delete 메서드를 호출해서 해당 단어를 삭제한다.
        # result에는 삭제된 행의 개수가 들어간다.
        if result > 0: # 삭제된 행의 개수가 0보다 크면 삭제 성공
            print("삭제되었습니다")
        else:
            print("삭제에 실패했습니다")

In [5]:
class Menu: # Menu 클래스는 사용자가 볼 메뉴 화면과 전체 프로그램 흐름을 담당한다.
    def __init__(self):
        # Menu 객체가 만들어질 때 실행되는 생성자이다.
        self.service = WordsService()
        # 실제 기능을 수행할 WordsService 객체를 생성한다.

    def run(self): # 메뉴 프로그램을 실행하는 메서드
        while True:
            try:
                print()
                print("===== 단어장 프로그램 =====")
                print("1. 등록하기")
                print("2. 출력하기")
                print("3. 검색하기")
                print("4. 수정하기")
                print("5. 삭제하기")
                print("6. 종료하기")
                menu = int(input("메뉴를 선택하세요"))

                if menu == 1:
                    print("단어를 등록합니다")
                    self.service.insert_word()
                    # WordsService의 insert_word 메서드를 호출해서 단어 등록 기능을 실행한다.
                elif menu == 2:
                    print("단어를 출력합니다")
                    self.service.print_all()
                    # WordsService의 print_all 메서드를 호출해서 전체 단어를 출력한다.
                elif menu == 3:
                    print("단어를 검색합니다")
                    self.service.search_words()
                    # WordsService의 search_words 메서드를 호출해서 단어 검색 기능을 실행한다.
                elif menu == 4:
                    print("단어를 수정합니다")
                    self.service.edit_word()
                    # WordsService의 edit_word 메서드를 호출해서 단어 수정 기능을 실행한다.
                elif menu == 5:
                    print("단어를 삭제합니다")
                    self.service.delete_word()
                    # WordsService의 delete_word 메서드를 호출해서 단어 삭제 기능을 실행한다.
                elif menu == 6:
                    print("프로그램을 종료합니다")
                    break
                    # while True 반복문을 종료한다.
                else:
                    print("메뉴는 1부터 6까지 입력해주세요")
            except Exception as e:
                print("오류: ", e)
                print("다시 입력하세요")

In [6]:
menu = Menu()
menu.run()


===== 단어장 프로그램 =====
1. 등록하기
2. 출력하기
3. 검색하기
4. 수정하기
5. 삭제하기
6. 종료하기
단어를 출력합니다
apple 뜻:사과, 레벨:1
avocado 뜻:아보카도, 레벨:2
banana 뜻:바나나, 레벨:1
orange 뜻:오렌지, 레벨:1

===== 단어장 프로그램 =====
1. 등록하기
2. 출력하기
3. 검색하기
4. 수정하기
5. 삭제하기
6. 종료하기
프로그램을 종료합니다
